# 📝 청킹 전략·RAPTOR 과제 LV1(기초)

한국어 백과 도입문 5편으로 청킹 전략의 규칙을 하나씩 확인합니다. 모델과 API 키 없이 풉니다.

맨 위 준비 셀을 차례로 실행한 뒤, 문항마다 답안 셀을 채우고 자가채점 셀로 확인하세요.

In [ ]:
# LV1은 모델 없이 분할기와 파이썬 계산만 씁니다.
import json
from pathlib import Path

import pandas as pd
from langchain_core.documents import Document

In [ ]:
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
import sys

material_dir = Path(".")
# 학생용·정답용 모두 실습자료 폴더의 util.py를 가져옵니다.
sys.path.insert(0, str(material_dir.resolve()))

data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )

In [ ]:
def make_documents(records):
    """원문 기록마다 본문과 출처(ID·제목·위치)를 담은 LangChain Document를 만듭니다."""
    return [
        Document(
            page_content=record["text"],
            # 분할한 뒤에도 source_id로 청킹 전 본문을 다시 찾습니다.
            metadata={
                "source_id": record["doc_id"],
                "title": record["title"],
                "url": record["url"],
                # 원문 전체 구간을 먼저 기록하고, 분할 뒤에는 각 청크의 구간으로 갱신합니다.
                "start_index": 0,
                "end_index": len(record["text"]),
            },
        )
        for record in records
    ]

In [ ]:
# 원문을 읽어 Document로 바꿉니다. 첫 원문(first_record)은 여러 문항에서 씁니다.
records = read_json("lv1_docs.json")
documents = make_documents(records)
first_record = records[0]

display(pd.DataFrame(records)[["doc_id", "title"]])

In [ ]:
# [제공 코드]
# 시작 위치와 청크 길이로 끝 위치를 기록하는 지원 함수입니다.
from util import set_end_offsets

청크의 원문 위치를 연결하는 보조 함수는 [util.py](util.py)에 있습니다. 교안·과제와 같은 폴더에 두고, 아래 표의 입력과 결과를 확인해 사용합니다.

| 제공 이름 | 내용 |
|---|---|
| `records` | 원문 딕셔너리 5개(`doc_id`, `title`, `text`) |
| `documents` | 같은 순서의 `Document` 리스트. metadata에 `source_id`·`start_index`·`end_index` |
| `first_record` | 첫 원문 딕셔너리(`wiki0`) |
| `set_end_offsets(chunks)` | 청크마다 `end_index = start_index + 글자 수`를 채워 돌려줍니다 |
| `to_index_documents(documents)` | `Document`를 LlamaIndex 파서 입력으로 바꿉니다(6번 앞에서 제공) |
| `evidence_recall(contexts, evidence)` | 반환 문서들이 근거 구간을 모두 덮은 비율(8번 앞에서 제공) |

문항별 입력(`neighbor_distances` 등)은 그 문항 바로 위 제공 코드 셀에 있습니다.

## 1. 고정 길이로 나누기

**배경**: 글자 수로만 자르면 문장 중간에도 경계가 생깁니다.

**요구사항**:

- **`fixed_chunks`** (`Document` 리스트): 첫 원문 `documents[:1]`을 문자 단위 Fixed 120자·겹침 0으로 나눈 결과. 공백은 지우지 않고, 청크마다 `start_index`·`end_index`를 기록합니다.

**확인 기준**: 청크는 3개이고 `start_index`는 `[0, 120, 240]`입니다. 이어 붙이면 첫 원문과 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문자 단위 Fixed 분할기로 자른 뒤 끝 위치를 채웁니다.

세부구현:
1. RecursiveCharacterTextSplitter에 크기·겹침을 주고, separators에는 빈 문자열 하나만 넣습니다.
2. strip_whitespace는 끄고 add_start_index는 켭니다.
3. split_documents 결과를 set_end_offsets에 넘깁니다.
```

</details>

In [ ]:
# [제공 코드]
# 고정 길이 분할에 사용할 분할기입니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(fixed_chunks, list) and fixed_chunks, (
    "fixed_chunks에 split_documents 결과(Document 리스트)를 담으세요."
)
assert "".join(doc.page_content for doc in fixed_chunks) == first_record["text"], (
    "청크를 이어 붙이면 첫 원문이어야 합니다. documents[:1]과 strip_whitespace=False를 확인하세요."
)

starts = [doc.metadata["start_index"] for doc in fixed_chunks]

assert starts == [0, 120, 240], (
    f"시작 위치가 {starts}입니다. 구분자를 빈 문자열 하나로 두고 add_start_index=True를 넣었는지 확인하세요."
)
assert all(doc.metadata["end_index"] == doc.metadata["start_index"] + len(doc.page_content) for doc in fixed_chunks), (
    "끝 위치가 갱신되지 않았습니다. set_end_offsets를 호출하세요."
)

## 2. 겹친 글자 확인하기

**배경**: 겹침을 주면 경계 글자가 두 청크에 남습니다.

**요구사항**:

- **`overlap_chunks`** (`Document` 리스트): 1번 설정에서 겹침만 20.
- **`left_overlap`** (문자열): 첫 청크의 마지막 20자.
- **`right_overlap`** (문자열): 둘째 청크의 처음 20자.

**확인 기준**: 두 문자열이 같고, 둘째 청크의 `start_index`는 100(=120-20)입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 크기에서 겹침을 뺀 위치에서 다음 청크가 시작합니다.

세부구현:
1. 1번 분할기 설정에서 chunk_overlap만 바꿔 다시 나눕니다.
2. 첫 청크의 끝과 둘째 청크의 처음을 슬라이싱으로 20자씩 꺼냅니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(overlap_chunks) >= 2, (
    "overlap_chunks에 청크가 두 개 이상 있어야 합니다. documents[:1]을 나눴는지 확인하세요."
)
assert overlap_chunks[1].metadata["start_index"] == 100, (
    "둘째 청크의 start_index가 100이 아닙니다. chunk_overlap=20과 add_start_index=True를 확인하세요."
)
assert len(left_overlap) == len(right_overlap) == 20, "문자열 슬라이싱으로 20자씩 꺼내세요."
assert left_overlap == right_overlap, "첫 청크는 끝 20자, 둘째 청크는 처음 20자를 꺼내야 두 문자열이 같습니다."

#### 제공 코드: 이웃 문장 거리

In [ ]:
# [제공 코드]
# 실제 임베딩 결과가 아니라 백분위 경계를 연습할 숫자입니다.
import numpy as np

# neighbor_distances[i]는 문장 i와 i+1 사이 거리입니다.
neighbor_distances = [0.10, 0.20, 0.30, 0.60, 0.80]
percentile = 50

## 3. Semantic의 백분위 경계 찾기

**배경**: Semantic 파서는 이웃 문장 사이 거리가 백분위 기준을 넘는 곳에서 자릅니다.

**요구사항**:

- **`distance_threshold`** (실수): `neighbor_distances`의 `percentile`(50) 백분위 값.
- **`boundary_indices`** (정수 리스트, 오름차순): 거리가 기준을 **초과**한 자리의 오른쪽 문장 번호. `neighbor_distances[i]`가 기준을 넘으면 `i + 1`을 담습니다.

**확인 기준**: 기준과 **같은** 거리는 경계가 아닙니다. 경계는 2개입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 백분위로 기준을 구하고, 기준보다 큰 거리의 위치를 모읍니다.

세부구현:
1. np.percentile로 distance_threshold를 구합니다.
2. enumerate로 위치와 거리를 함께 돌며 기준보다 큰 거리만 고릅니다.
3. 위치에 1을 더해 boundary_indices에 담습니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(distance_threshold - 0.30) < 1e-9, "np.percentile에 neighbor_distances와 percentile을 넣어 기준을 구하세요."
assert boundary_indices == [4, 5], (
    f"경계가 {boundary_indices}입니다. 기준과 같은 거리(0.30)는 경계가 아니므로 >로 비교하고, 위치에 1을 더한 오른쪽 문장 번호를 담으세요."
)

## 4. 백분위와 경계 수

**배경**: 백분위가 높을수록 기준도 큽니다.

**요구사항**:

- **`boundary_counts`** (딕셔너리): 3번 거리에서 백분위 50·80 기준을 초과한 거리 수(키는 정수 백분위).
- 서술 칸: 청크가 늘어도 검색 품질이 꼭 좋아지지 않는 이유 1~2문장.

**확인 기준**: 백분위를 높이면 경계 수는 같거나 줄어듭니다.

<details><summary>힌트</summary>

```text
접근방법:
- 3번 계산을 백분위 두 개로 반복합니다.

세부구현:
1. 백분위 50과 80을 차례로 돌며 np.percentile로 기준을 구합니다.
2. 기준보다 큰 거리 개수를 세어 백분위를 키로 담습니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(boundary_counts, dict) and set(boundary_counts) == {50, 80}, (
    "boundary_counts의 키는 정수 50과 80입니다."
)
assert boundary_counts == {50: 2, 80: 1}, (
    f"결과가 {boundary_counts}입니다. 백분위마다 기준을 새로 구하고, 기준을 초과(>)한 거리만 세세요."
)

*(여기에 1~2문장으로 서술하세요: 청크가 많아질 때 생기는 문제 하나)*

#### 제공 코드: 자식이 가리킨 부모 ID

In [ ]:
# [제공 코드]
# 검색된 자식 6개가 가리킨 부모 ID입니다. 같은 부모가 여러 번 나옵니다.
selected_children = [
    {"parent_id": records[4]["doc_id"]},
    {"parent_id": records[0]["doc_id"]},
    {"parent_id": records[4]["doc_id"]},
    {"parent_id": records[3]["doc_id"]},
    {"parent_id": records[0]["doc_id"]},
    {"parent_id": records[1]["doc_id"]},
]

## 5. 검색된 자식의 부모 ID 모으기

**배경**: 여러 자식이 같은 부모를 가리킬 수 있습니다.

**요구사항**:

- **`parent_ids`** (문자열 리스트): `selected_children`의 `parent_id`를 처음 나온 순서로 중복 없이(정렬 안 함).

**확인 기준**: 부모 4개, 첫 원소는 `selected_children[0]["parent_id"]`.

<details><summary>힌트</summary>

```text
접근방법:
- 이미 담은 ID인지 확인하며 리스트에 추가합니다.

세부구현:
1. 빈 리스트를 만듭니다.
2. selected_children을 차례로 돌며 parent_id가 리스트에 없을 때만 append합니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(parent_ids, list), "parent_ids는 리스트여야 합니다. set은 순서를 보장하지 않습니다."
assert len(parent_ids) == len(set(parent_ids)) == 4, "중복을 없애면 부모는 4개입니다."
assert parent_ids == [records[4]["doc_id"], records[0]["doc_id"], records[3]["doc_id"], records[1]["doc_id"]], (
    "처음 나온 순서를 유지하세요. sorted()나 set()은 순서를 바꿉니다."
)

#### 제공 코드: 문장 파서와 입력 변환

In [ ]:
# [제공 코드]
# 모델을 호출하지 않는 문장 파서입니다.
from llama_index.core import Document as IndexDocument
from llama_index.core.node_parser import SentenceWindowNodeParser

In [ ]:
# [제공 코드]
def to_index_documents(documents):
    """LangChain Document를 LlamaIndex 파서가 읽는 Document로 바꿉니다."""
    # 본문·metadata를 유지하면서 파서가 사용하는 Document 형식으로 바꿉니다.
    result = [IndexDocument.from_langchain_format(doc) for doc in documents]
    for doc in result:
        # 분할 노드의 ref_doc_id로 청킹 전 원문을 찾을 수 있도록 ID를 맞춥니다.
        doc.id_ = doc.metadata["source_id"]
        # metadata는 보관하되 모델 입력에서 빼서 긴 URL 등이 분할·의미 비교에 섞이지 않게 합니다.
        doc.excluded_embed_metadata_keys = list(doc.metadata)
        doc.excluded_llm_metadata_keys = list(doc.metadata)

    return result

## 6. 첫 문장의 주변 문맥 확인하기

**배경**: Sentence Window는 한 문장으로 검색하고, 답변에는 앞뒤 문장을 붙인 window를 넘깁니다.

**요구사항**:

- **`window_parser`**: `window_size=1`인 `SentenceWindowNodeParser`.
- **`first_nodes`** (노드 리스트): 첫 원문(`documents[:1]`)만 파서에 넣어 얻은 문장 노드.
- **`start_window`** (문자열): 첫 노드의 `metadata["window"]`. 첫 노드의 `text`와 함께 출력하세요.

**확인 기준**: 모든 노드의 `source_id`는 `wiki0`입니다. 첫 문장은 앞 문장이 없으므로 window는 첫째·둘째 문장뿐입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 원문을 파서 입력으로 바꾼 뒤 문장 노드를 만듭니다.

세부구현:
1. SentenceWindowNodeParser.from_defaults에 window_size만 줍니다.
2. to_index_documents로 바꾼 첫 원문을 get_nodes_from_documents에 넘깁니다.
3. 첫 노드의 metadata에서 window를 꺼냅니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert all(node.metadata["source_id"] == first_record["doc_id"] for node in first_nodes), (
    "첫 원문만 넣으세요. documents[:1]입니다."
)
assert len(first_nodes) >= 2, "파서가 첫 원문을 문장 노드로 나눴는지 확인하세요."
assert start_window == first_nodes[0].metadata["window"], 'start_window에는 첫 노드의 metadata["window"]를 담으세요.'
assert start_window == " ".join(node.text for node in first_nodes[:2]), (
    "window_size=1이면 첫 문장의 window는 첫째·둘째 문장뿐입니다. window_size를 확인하세요."
)

#### 제공 코드: 한 부모의 검색된 자식 ID

In [ ]:
# [제공 코드]
# 전체 자식이 4개인 부모에서 검색된 자식 ID입니다(같은 자식이 두 번 검색됨).
retrieved_child_ids = ["child_0", "child_0", "child_1"]
total_children = 4
merge_threshold = 0.5

## 7. 병합 비율과 병합 여부 판단하기

**배경**: Auto-merging은 같은 부모의 자식이 기준을 초과해 검색될 때만 부모로 바꿉니다.

**요구사항**:

- **`unique_child_ids`** (집합): `retrieved_child_ids`에서 중복을 없앤 자식 ID.
- **`hit_ratio`** (실수): 고유 자식 수 ÷ `total_children`(이 부모의 전체 자식 수).
- **`should_merge`** (불리언): `hit_ratio`가 `merge_threshold`를 **초과**하면 True. 비율과 함께 출력하세요.

**확인 기준**: 비율이 기준과 같으면 병합하지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 자식은 한 번만 세고, 분모는 이 부모의 전체 자식 수입니다.

세부구현:
1. set으로 중복을 없앱니다.
2. 고유 개수를 total_children으로 나눕니다.
3. merge_threshold와 초과로 비교합니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert unique_child_ids == {"child_0", "child_1"}, "retrieved_child_ids를 set으로 바꿔 중복을 없애세요."
assert hit_ratio == 0.5, (
    f"hit_ratio가 {hit_ratio}입니다. 고유 자식 수를 total_children으로 나누세요. 검색 결과 수로 나누지 않습니다."
)
assert should_merge is False, "비율 0.5는 기준 0.5를 초과하지 않습니다. >=가 아니라 >로 비교하세요."

#### 제공 코드: 근거 구간 Recall

In [ ]:
# [제공 코드]
def evidence_recall(contexts, evidence):
    """반환한 원문 구간들이 고정 근거를 얼마나 회수했는지 계산합니다.

    Args:
        contexts: source_id, start_index, end_index를 가진 원문 Document 리스트.
        evidence: doc_id, start, end를 가진 비어 있지 않은 근거 리스트.
    Returns:
        전체 위치가 회수된 근거 수 / 전체 근거 수. 답변 정확도는 아닙니다.
    """
    covered = {}
    for context in contexts:
        meta = context.metadata
        positions = covered.setdefault(meta["source_id"], set())
        # 출처별 문자 위치의 합집합이므로 겹친 청크가 근거 수를 늘리지 않습니다.
        positions.update(range(meta["start_index"], meta["end_index"]))

    found = 0
    for item in evidence:
        required = set(range(item["start"], item["end"]))
        # 다른 원문의 같은 위치는 인정하지 않으며, 근거 일부만 덮은 경우도 제외합니다.
        if required.issubset(covered.get(item["doc_id"], set())):
            found += 1

    return found / len(evidence)

#### 제공 코드: 두 청크에 나뉜 근거

In [ ]:
# [제공 코드]
# 첫 원문의 근거 구간 하나와, 그 근거를 나눠 덮는 두 청크입니다.
source_id = first_record["doc_id"]
# end는 포함하지 않으므로 start=17, end=48은 17~47번 글자입니다.
evidence = [{"doc_id": source_id, "start": 17, "end": 48}]

fragment_contexts = [
    Document(
        page_content=first_record["text"][10:35],
        metadata={"source_id": source_id, "start_index": 10, "end_index": 35},
    ),
    Document(
        page_content=first_record["text"][35:60],
        metadata={"source_id": source_id, "start_index": 35, "end_index": 60},
    ),
]

print("근거:", first_record["text"][17:48])

## 8. 두 청크가 함께 회수한 근거

**배경**: 근거가 두 청크에 나뉘어도 합쳐 모두 덮으면 회수입니다.

**요구사항**:

- **`covered`** (정수 집합): 두 청크의 `range(start_index, end_index)` 합집합.
- **`first_only_recall`** (실수): 첫 청크만 넘긴 `evidence_recall`.
- **`span_recall`** (실수): 두 청크를 넘긴 `evidence_recall`.

**확인 기준**: 첫 청크만 0.0, 두 청크 1.0입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 청크마다 위치 범위를 만들어 합치고, 제공 함수로 두 경우를 계산합니다.

세부구현:
1. 각 청크 metadata의 두 위치로 range를 만들어 집합에 더합니다.
2. fragment_contexts[:1]과 fragment_contexts를 각각 evidence_recall에 넘깁니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert covered == set(range(10, 60)), "두 청크(10~35, 35~60)의 문자 위치를 합집합으로 모으세요."
assert first_only_recall == 0.0, "첫 청크만(fragment_contexts[:1]) 넘기세요. 근거 일부만 덮으면 0입니다."
assert span_recall == 1.0, "두 청크를 함께 넘긴 evidence_recall 결과를 담으세요."

#### 제공 코드: 요약 노드의 원문 연결

In [ ]:
# [제공 코드]
# L1·L2 요약 노드의 leaf_ids는 그 요약의 입력이 된 원문 잎(L0) ID입니다.
summary_nodes = {
    "summary_1_0": {"level": 1, "leaf_ids": ["leaf_0", "leaf_2"]},
    "summary_1_1": {"level": 1, "leaf_ids": ["leaf_1", "leaf_3", "leaf_4"]},
    "summary_2_0": {"level": 2, "leaf_ids": ["leaf_0", "leaf_1", "leaf_2", "leaf_3", "leaf_4"]},
}

# 모든 계층을 한 인덱스에서 검색한 결과의 ID 순서입니다. 요약과 잎이 섞여 있습니다.
retrieved_ids = ["summary_1_1", "leaf_3", "leaf_2"]

## 9. 요약 노드에서 원문 잎 찾기

**배경**: 요약이 검색돼도 답변 근거로는 그 요약의 원문 잎(L0)을 넘깁니다.

**요구사항**:

- **`evidence_ids`** (문자열 리스트): `retrieved_ids`를 차례로 보며 요약이면 그 `leaf_ids`를, 잎이면 자기 ID를 담습니다. 처음 나온 순서로, 중복 없이.

**확인 기준**: 잎 ID 4개, 첫 원소는 `leaf_1`입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 요약은 leaf_ids로 펼치고, 5번처럼 중복 없이 순서대로 모읍니다.

세부구현:
1. 빈 리스트를 만듭니다.
2. retrieved_ids를 돌며 summary_nodes에 있으면 그 leaf_ids를, 없으면 자기 ID 하나를 꺼냅니다.
3. 꺼낸 잎 ID가 리스트에 없을 때만 추가합니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(evidence_ids, list) and all(item.startswith("leaf_") for item in evidence_ids), (
    "evidence_ids에는 요약 ID가 아니라 원문 잎 ID만 담습니다."
)
assert len(evidence_ids) == len(set(evidence_ids)) == 4, "leaf_3처럼 두 번 나온 잎은 한 번만 담으세요."
assert evidence_ids[0] == "leaf_1" and evidence_ids[-1] == "leaf_2", (
    "retrieved_ids 순서대로 담으세요. 요약은 leaf_ids 순서대로 펼칩니다."
)

## 10. 요약 트리의 두 연결 설명하기

**배경**: `child_ids`와 `leaf_ids`는 쓰임이 다릅니다.

**요구사항**:

L2 요약의 `child_ids`(바로 아래 L1)와 `leaf_ids`(원문 잎 L0)를 언제 쓰는지 한 문장씩, 출처가 연결돼도 원문 사실이 요약에 다 남았다고 볼 수 없는 이유를 한 문장으로 쓰세요.

<details><summary>힌트</summary>

```text
접근방법:
- 직접 요약 입력과 최종 원문 출처를 나누어 설명합니다.
```

</details>

*(여기에 3문장으로 서술하세요: child_ids의 쓰임, leaf_ids의 쓰임, 출처 연결과 내용 보존의 차이)*